In [ ]:
!pip install -q transformers datasets

In [ ]:
!pip install -q bert-score

In [ ]:
!pip install google-generativeai

In [ ]:
!pip install -q chromadb sentence-transformers


In [ ]:
import google.generativeai as genai

# 1. Leggi la chiave
from kaggle_secrets import UserSecretsClient
GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise RuntimeError("❌ API key Gemini non trovata. Controlla i Secrets di Kaggle.")

# 2. Configura
GEMINI_MODEL = "gemini-2.5-flash"
genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel(GEMINI_MODEL)
print(f"Gemini configurato: {GEMINI_MODEL}")

# 3. Test rapido
try:
    response = gemini_model.generate_content("Reply with OK")
    print("✅ Gemini funziona:", response.text.strip())
except Exception as e:
    raise RuntimeError(f"❌ Chiave trovata ma chiamata fallita: {e}")

In [ ]:
import google.generativeai as genai
import time
 
CONFIDENCE_THRESHOLD_CLOSED = 0.80   
CONFIDENCE_THRESHOLD_OPEN   = 0.50   


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoImageProcessor
from PIL import Image
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertModel, ViTModel, AutoTokenizer, ViTImageProcessor
from datasets import load_dataset
import numpy as np
from transformers import pipeline
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoImageProcessor


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
dataset_path = "/kaggle/input/datasets/angelo01paldino/vqarad-final/VQA_RAD Output FolderValidation" 
hf_dataset = load_from_disk(dataset_path)
print("Partizioni trovate nel dataset:", hf_dataset.keys())

print("Inizializzazione Tokenizer e Image Processor...")
tokenizer = AutoTokenizer.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
image_processor = AutoImageProcessor.from_pretrained("google/vit-large-patch32-384")
tokenizer.truncation_side = "left"

In [ ]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer
 
# ── Parametri RAG ──────────────────────────────────────────────────────────
RAG_K              = 5          # coppie da recuperare per ogni domanda
RAG_ENCODER        = "pritamdeka/S-PubMedBert-MS-MARCO"  # encoder biomedico leggero
CHROMA_PATH        = "/kaggle/working/chroma_vqa_index"  # indice persistente su disco
 
def build_rag_index(hf_dataset, encoder_model, chroma_path=CHROMA_PATH):
    """
    Costruisce (o ricarica se già esistente) l'indice ChromaDB con tutte
    le coppie Q&A di training + validation set.
 
    Ogni documento nell'indice contiene:
      - document : testo della domanda contestualizzata
      - metadata : answer, organ, question_type, answer_type
      - embedding: vettore della domanda (calcolato da encoder_model)
 
    Il filtro per organo e tipo domanda viene applicato al momento
    del retrieval tramite i metadata ChromaDB (where clause).
    """
    chroma_client = chromadb.PersistentClient(path=chroma_path)
 
    # Se la collection esiste già, non ricalcolare tutto
    existing = [c.name for c in chroma_client.list_collections()]
    if "vqa_rad_pairs" in existing:
        print("Indice RAG già presente → caricamento da disco.")
        collection = chroma_client.get_collection("vqa_rad_pairs")
        return chroma_client, collection
 
    print("Costruzione indice RAG (training + validation)...")
    collection = chroma_client.create_collection(
        name="vqa_rad_pairs",
        metadata={"hnsw:space": "cosine"},  # similarità coseno
    )
 
    # Raccogliamo tutte le coppie da train + validation
    splits_to_index = ["train"]
    all_ids, all_docs, all_embeddings, all_metadata = [], [], [], []
 
    for split_name in splits_to_index:
        split = hf_dataset[split_name]
        questions, metadata_list = [], []
 
        for idx, item in enumerate(split):
            organ      = str(item.get('image_organ',   'UNKNOWN')).strip().upper()
            q_type     = str(item.get('question_type', 'UNKNOWN')).strip().upper()
            ans_type   = str(item.get('answer_type',   'CLOSED')).strip().upper()
            question   = str(item['question'])
            answer     = str(item['answer'])
 
            contextualized_q = f"[{organ} | {q_type}] {question}"
 
            questions.append(contextualized_q)
            metadata_list.append({
                "answer":      answer,
                "organ":       organ,
                "q_type":      q_type,
                "ans_type":    ans_type,
                "split":       split_name,
            })
            all_ids.append(f"{split_name}_{idx}")
            all_docs.append(contextualized_q)
            all_metadata.append(metadata_list[-1])
 
        # Calcolo embeddings in batch (più efficiente)
        print(f"  Encoding {len(questions)} domande ({split_name})...")
        embeddings = encoder_model.encode(questions, batch_size=64,
                                          show_progress_bar=True,
                                          normalize_embeddings=True)
        all_embeddings.extend(embeddings.tolist())
 
    # Inserimento in ChromaDB in batch da 500 (limite ChromaDB)
    batch_size = 500
    for i in range(0, len(all_ids), batch_size):
        collection.add(
            ids        = all_ids[i:i+batch_size],
            documents  = all_docs[i:i+batch_size],
            embeddings = all_embeddings[i:i+batch_size],
            metadatas  = all_metadata[i:i+batch_size],
        )
 
    print(f"Indice RAG costruito: {collection.count()} documenti totali.")
    return chroma_client, collection
 
 
def retrieve_similar_pairs(query_question, organ, ans_type,
                            collection, encoder_model, k=RAG_K):
    """
    Recupera le K coppie Q&A più simili alla domanda corrente,
    filtrando per stesso organo E stesso tipo di risposta (OPEN/CLOSED).
 
    Restituisce una stringa formattata pronta per la concatenazione:
      '[CONTEXT] Q: ... A: ... | Q: ... A: ... [/CONTEXT]'
    """
    query_embedding = encoder_model.encode(
        query_question,
        normalize_embeddings=True
    ).tolist()
 
    # Filtro metadata: stesso organo E stesso tipo domanda
    where_filter = {
        "$and": [
            {"organ":    {"$eq": organ}},
            {"ans_type": {"$eq": ans_type}},
        ]
    }
 
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=where_filter,
        include=["documents", "metadatas", "distances"],
    )
 
    # Costruzione del contesto testuale
    context_parts = []
    for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
        # Estraiamo solo la domanda (senza il prefisso [ORGAN | TYPE])
        raw_question = doc.split("] ", 1)[-1] if "] " in doc else doc
        context_parts.append(f"Q: {raw_question} A: {meta['answer']}")
 
    if not context_parts:
        return ""  # nessun risultato → nessun contesto aggiunto
 
    return "[CONTEXT] " + " | ".join(context_parts) + " [/CONTEXT]"
 
 
# ── Inizializzazione (eseguire dopo aver caricato hf_dataset) ───────────────
print("Caricamento encoder RAG...")
rag_encoder = SentenceTransformer(
    RAG_ENCODER,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)
 
chroma_client, rag_collection = build_rag_index(hf_dataset, rag_encoder)
print("Sistema RAG pronto.")

In [ ]:
from datasets import load_dataset
 
# ── Parametri KB esterna ───────────────────────────────────────────────────
KB_K           = 5                        # chunk KB per ogni domanda
KB_CHROMA_PATH = "/kaggle/working/chroma_kb_index"  # collezione separata
KB_MAX_PUBMED  = 3000                      # documenti da pubmed_qa
KB_MAX_MEDMCQA = 3000                    # documenti da medmcqa
 
 
def _format_pubmed_doc(item):
    """
    Da pubmed_qa estrae:
      - question  : la domanda clinica
      - context   : primo abstract disponibile (contesto scientifico)
      - long_answer: risposta articolata
 
    Restituisce un unico testo concatenato da indicizzare.
    Il formato privilegia la risposta lunga perché è il contenuto
    più informativo per il retrieval clinico.
    """
    question    = str(item.get("question", "")).strip()
    long_answer = str(item.get("long_answer", "")).strip()
 
    # context è un dict con chiave 'contexts' (lista di stringhe)
    contexts = item.get("context", {})
    if isinstance(contexts, dict):
        ctx_list = contexts.get("contexts", [])
        context_text = " ".join(ctx_list[:2]) if ctx_list else ""
    else:
        context_text = ""
 
    # Documento finale: domanda + primo frammento di contesto + risposta
    doc = f"Q: {question}"
    if context_text:
        # Tronchiamo il contesto a 200 caratteri per non dominare il testo
        doc += f" CONTEXT: {context_text[:200]}"
    if long_answer:
        doc += f" A: {long_answer}"
 
    return doc.strip()
 
 
def _format_medmcqa_doc(item):
    """
    Da medmcqa estrae:
      - question : domanda clinica
      - exp      : spiegazione della risposta corretta (il più informativo)
      - opX      : opzione corretta (cop indica l'indice: 0=a,1=b,2=c,3=d)
 
    Restituisce un unico testo concatenato da indicizzare.
    La spiegazione (exp) è il contenuto più prezioso perché ragiona
    sul perché la risposta è corretta — esattamente il tipo di
    conoscenza clinica che vogliamo trasferire al modello VQA.
    """
    question    = str(item.get("question", "")).strip()
    explanation = str(item.get("exp", "")).strip()
 
    # Recupera l'opzione corretta
    cop_idx     = item.get("cop", 0)  # 0=a, 1=b, 2=c, 3=d
    options     = ["opa", "opb", "opc", "opd"]
    correct_key = options[cop_idx] if cop_idx < len(options) else "opa"
    correct_ans = str(item.get(correct_key, "")).strip()
 
    doc = f"Q: {question}"
    if correct_ans:
        doc += f" A: {correct_ans}"
    if explanation:
        doc += f" EXPLANATION: {explanation}"
 
    return doc.strip()
 
 
def build_kb_index(rag_encoder, kb_chroma_path=KB_CHROMA_PATH,
                   max_pubmed=KB_MAX_PUBMED, max_medmcqa=KB_MAX_MEDMCQA):
    """
    Costruisce (o ricarica se già esistente) la collezione ChromaDB
    con i documenti della KB esterna (pubmed_qa + medmcqa).
 
    Collezione separata da quella della Strategia 2:
      - vqa_rad_pairs   → Strategia 2 (coppie Q&A del dataset)
      - medical_kb      → Strategia 1 (KB esterna PubMed + MedMCQA)
 
    Ogni documento ha metadata:
      - source : "pubmed_qa" o "medmcqa"
    Non ha filtri per organo (i documenti esterni non hanno quel metadato):
    il retrieval avviene per similarità semantica pura.
    """
    kb_client = chromadb.PersistentClient(path=kb_chroma_path)
 
    existing = [c.name for c in kb_client.list_collections()]
    if "medical_kb" in existing:
        print("KB esterna già presente → caricamento da disco.")
        kb_collection = kb_client.get_collection("medical_kb")
        return kb_client, kb_collection
 
    print("Costruzione KB esterna (pubmed_qa + medmcqa)...")
    kb_collection = kb_client.create_collection(
        name="medical_kb",
        metadata={"hnsw:space": "cosine"},
    )
 
    all_ids, all_docs, all_embeddings, all_metadata = [], [], [], []
 
    # ── 1. PubMed QA ────────────────────────────────────────────────────────
    print(f"  Caricamento pubmed_qa (max {max_pubmed} documenti)...")
    pubmed_ds = load_dataset("pubmed_qa", "pqa_labeled",
                             split="train", trust_remote_code=True)
 
    for idx, item in enumerate(pubmed_ds):
        if idx >= max_pubmed:
            break
        doc = _format_pubmed_doc(item)
        if not doc:
            continue
        all_ids.append(f"pubmed_{idx}")
        all_docs.append(doc)
        all_metadata.append({"source": "pubmed_qa"})
 
    print(f"  → {len(all_docs)} documenti PubMed pronti.")
 
    # ── 2. MedMCQA ──────────────────────────────────────────────────────────
    print(f"  Caricamento medmcqa (max {max_medmcqa} documenti)...")
    medmcqa_ds = load_dataset("medmcqa", split="train")
 
    pubmed_count = len(all_docs)
    for idx, item in enumerate(medmcqa_ds):
        if idx >= max_medmcqa:
            break
        doc = _format_medmcqa_doc(item)
        if not doc:
            continue
        all_ids.append(f"medmcqa_{idx}")
        all_docs.append(doc)
        all_metadata.append({"source": "medmcqa"})
 
    print(f"  → {len(all_docs) - pubmed_count} documenti MedMCQA pronti.")
    print(f"  Totale KB: {len(all_docs)} documenti.")
 
    # ── Encoding in batch ───────────────────────────────────────────────────
    print("  Encoding KB (potrebbe richiedere qualche minuto)...")
    embeddings = rag_encoder.encode(
        all_docs,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    all_embeddings = embeddings.tolist()
 
    # ── Inserimento in ChromaDB (batch da 500) ──────────────────────────────
    batch_size = 500
    for i in range(0, len(all_ids), batch_size):
        kb_collection.add(
            ids        = all_ids[i:i+batch_size],
            documents  = all_docs[i:i+batch_size],
            embeddings = all_embeddings[i:i+batch_size],
            metadatas  = all_metadata[i:i+batch_size],
        )
 
    print(f"KB esterna costruita: {kb_collection.count()} documenti totali.")
    return kb_client, kb_collection
 
 
def retrieve_kb_context(query_question, kb_collection, rag_encoder, k=KB_K):
    """
    Recupera i K documenti più rilevanti dalla KB esterna
    per la domanda corrente.
 
    A differenza della Strategia 2, NON filtra per organo:
    la KB esterna non ha quel metadato, e vogliamo recuperare
    conoscenza clinica generale rilevante semanticamente.
 
    Restituisce una stringa formattata:
      '[KB] testo1 | testo2 [/KB]'
    """
    query_embedding = rag_encoder.encode(
        query_question,
        normalize_embeddings=True,
    ).tolist()
 
    results = kb_collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
 
    kb_parts = []
    for doc in results["documents"][0]:
        # Tronchiamo ogni chunk a 150 caratteri per non saturare max_len
        kb_parts.append(doc[:150].strip())
 
    if not kb_parts:
        return ""
 
    return "[KB] " + " | ".join(kb_parts) + " [/KB]"
 
 
# ── Inizializzazione KB (dopo aver già inizializzato rag_encoder) ───────────
# NOTA: rag_encoder è già definito nella Cella 3b, lo riusiamo qui.
kb_client, kb_collection = build_kb_index(rag_encoder)
print("KB esterna pronta.")

In [ ]:
import time
import threading

_gemini_call_times = []
_gemini_lock = threading.Lock()  # thread-safe per sicurezza
GEMINI_MAX_PER_MIN = 10


def _gemini_rate_limit(max_per_minute=GEMINI_MAX_PER_MIN):
    with _gemini_lock:
        while True:
            now = time.time()
            _gemini_call_times[:] = [t for t in _gemini_call_times if now - t < 60]
            
            if len(_gemini_call_times) < max_per_minute:
                _gemini_call_times.append(now)
                return
            
            oldest = _gemini_call_times[0]
            wait   = 60.0 - (now - oldest) + 0.5
            wait   = max(wait, 0.1)
            print(f"  [Rate limiter] {len(_gemini_call_times)}/min raggiunto, "
                  f"attendo {wait:.1f}s...")
            time.sleep(wait)  


In [ ]:


# ==========================================
# 1. CARICAMENTO DATASET HUGGING FACE
# ==========================================

# ==========================================
# 2. CLASSE PYTORCH AGGIORNATA
# ==========================================
class HFVQARADDataset(Dataset):
    def __init__(self, hf_dataset_split, tokenizer, image_processor, max_len=512, rag_collection=None, rag_encoder=None,kb_collection=None, kb_encoder=None):
        self.dataset = hf_dataset_split
        self.tokenizer = tokenizer
        self.image_processor = image_processor
        self.max_len = max_len
        self.rag_collection = rag_collection   
        self.rag_encoder = rag_encoder 
        self.kb_collection   = kb_collection   
        self.kb_encoder      = kb_encoder       

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # 1. Immagine 
        image = item['image'].convert("RGB")
        
        original_question = str(item['question'])
        answer = str(item['answer'])
        
        # 2. Estrazione Metadati 
        ans_type_str = str(item.get('answer_type', 'CLOSED')).strip().upper()
        # Maschera binaria: 0 per le chiuse (Sì/No), 1 per le aperte
        question_type_idx = 0 if ans_type_str == "CLOSED" else 1 
        
        q_type_category = str(item.get('question_type', 'UNKNOWN')).strip().upper()
        organ_category = str(item.get('image_organ', 'UNKNOWN')).strip().upper()



        # 3. HIERARCHICAL PROMPTING (Iniezione del Contesto)
        contextualized_question = f"[{organ_category} | {q_type_category}] {original_question}"
        
        # ── MODIFICA: arricchimento RAG ─────────────────────────────────────
        # Se il sistema RAG è disponibile, recupera le K coppie simili
        # e le prepende alla domanda come contesto testuale.
        # Se RAG non è disponibile (rag_collection=None), si comporta
        # esattamente come l'originale → retrocompatibile.
        
        rag_context = ""
        if self.rag_collection is not None and self.rag_encoder is not None:
            rag_context = retrieve_similar_pairs(
                query_question = contextualized_question,
                organ          = organ_category,
                ans_type       = ans_type_str,
                collection     = self.rag_collection,
                encoder_model  = self.rag_encoder,
                k              = RAG_K,
            )


        # ── contesto da KB esterna ─────────────────────────────
        kb_context = ""
        if self.kb_collection is not None and self.kb_encoder is not None:
            kb_context = retrieve_kb_context(
                query_question = contextualized_question,
                kb_collection  = self.kb_collection,
                rag_encoder    = self.kb_encoder,
                k              = KB_K,
            )
            
        # ── fine modifica RAG ────────────────────────────────────────────────
        # ── Costruzione testo finale ─────────────────────────────────────────
        # Ordine: [CONTEXT Strategia2] [KB Strategia1] [domanda originale]
        # Il tokenizer troncherà a max_len=128 se necessario.
        # In caso di troncamento, il modello perde i chunk meno recenti
        # (quelli a sinistra) → la domanda originale è sempre preservata
        # mettendola alla fine.
        parts = [p for p in [rag_context, kb_context, contextualized_question] if p]
        final_question = " ".join(parts)
 
        pixel_values = self.image_processor(image, return_tensors="pt").pixel_values.squeeze(0)
 
        q_tokens = self.tokenizer(final_question, truncation=True,
                                  padding='max_length', max_length=self.max_len,
                                  return_tensors="pt")
        if q_tokens.input_ids.size(1) > self.max_len:
            q_tokens["input_ids"]      = q_tokens.input_ids[:, -self.max_len:]
            q_tokens["attention_mask"] = q_tokens.attention_mask[:, -self.max_len:]
        a_tokens = self.tokenizer(answer, truncation=True,
                                  padding='max_length', max_length=self.max_len,
                                  return_tensors="pt")

        

        

        return {
            "pixel_values": pixel_values,
            "input_ids": q_tokens.input_ids.squeeze(0),
            "attention_mask": q_tokens.attention_mask.squeeze(0),
            "labels": a_tokens.input_ids.squeeze(0),
            "question_type": torch.tensor(question_type_idx, dtype=torch.long), # Va al modello per il routing
            "ans_type_str": ans_type_str,   # "OPEN" / "CLOSED" per l'analisi
            "q_type_str": q_type_category,  # "PRES", "POS", ecc. per l'analisi
            "organ_str": organ_category,    # "HEAD", "CHEST", ecc. per l'analisi
            "original_q": original_question, # Salviamo la domanda per le stampe finali
            "dataset_idx": idx
        }


train_set = HFVQARADDataset(
    hf_dataset['train'], tokenizer, image_processor,
    rag_collection = rag_collection,
    rag_encoder    = rag_encoder,
    kb_collection  = kb_collection,   
    kb_encoder     = rag_encoder,     
)
val_set = HFVQARADDataset(
    hf_dataset['validation'], tokenizer, image_processor,
    rag_collection = rag_collection,
    rag_encoder    = rag_encoder,
    kb_collection  = kb_collection,   
    kb_encoder     = rag_encoder,     
)
test_set = HFVQARADDataset(
    hf_dataset['test'], tokenizer, image_processor,
    rag_collection = rag_collection,
    rag_encoder    = rag_encoder,
    kb_collection  = kb_collection,   
    kb_encoder     = rag_encoder,     
)
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=8, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=8, shuffle=False)
 
print(f"Campioni pronti -> Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")

In [ ]:
CLOSED_CLASSES = ["yes", "no"]
YES_TOKEN_ID = tokenizer.convert_tokens_to_ids("yes")
NO_TOKEN_ID  = tokenizer.convert_tokens_to_ids("no")
assert YES_TOKEN_ID != tokenizer.unk_token_id, "‘yes’ non è un token singolo"
assert NO_TOKEN_ID  != tokenizer.unk_token_id, "‘no’ non è un token singolo"
IDX_TO_TOKEN = {0: YES_TOKEN_ID, 1: NO_TOKEN_ID}

In [ ]:
class CustomMedVQAModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=1024):
        super().__init__()
        
        self.vision_encoder = ViTModel.from_pretrained("google/vit-large-patch32-384")
        self.bert_encoder = BertModel.from_pretrained("emilyalsentzer/Bio_ClinicalBERT")
        for param in self.bert_encoder.parameters():
            param.requires_grad = False  # frozen come il resto di BERT
        
        # --- TECNICA: FREEZING ---
        # Blocchiamo i gradienti per l'encoder visivo per preservare i pesi pre-addestrati
        for param in self.vision_encoder.parameters():
            param.requires_grad = False

        n_layers_to_unfreeze = 3
        for layer in self.vision_encoder.encoder.layer[-n_layers_to_unfreeze:]:
            for param in layer.parameters():
                param.requires_grad = True
        
        for param in self.vision_encoder.layernorm.parameters():
            param.requires_grad = True
        
        # 2. Embedding Linguistici (Bio_ClinicalBERT)
        self.embedding = self.bert_encoder.embeddings.word_embeddings 
        self.vision_projection = nn.Linear(1024, 768)
        self.cross_attention = nn.MultiheadAttention(embed_dim=768, num_heads=8, batch_first=True)
        self.layer_norm = nn.LayerNorm(768)

        self.pre_head_dropout = nn.Dropout(0.2)

        # --- HEAD 1: Domande Chiuse (MLP) ---
        self.closed_head = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2) 
        )

        # --- HEAD 2: Domande Aperte (Decoder) ---
        decoder_layer = nn.TransformerDecoderLayer(d_model=768, nhead=8, batch_first=True, dropout=0.27150897038028765)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=3)
        
        self.fc_out = nn.Linear(768, vocab_size)
        self.fc_out.weight = self.embedding.weight


    def forward(self, pixel_values, question_ids, answer_ids, attention_mask=None):
        
        # Estrazione feature e Cross-Attention Fusion
        vision_feats = self.vision_encoder(pixel_values).last_hidden_state
        vision_feats = self.vision_projection(vision_feats)
        bert_output = self.bert_encoder(
            input_ids=question_ids,
            attention_mask=attention_mask
        )
        question_feats = bert_output.last_hidden_state
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()  # (B, seq_len, 1)
            question_feats = question_feats * mask
        
        attn_output, _ = self.cross_attention(query=question_feats, key=vision_feats, value=vision_feats)
        memory = self.layer_norm(question_feats + attn_output)
        
        # ── Dropout pre-head ──────────────────────────────────────────────
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (memory * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            pooled = memory.mean(dim=1)
        pooled_memory = self.pre_head_dropout(pooled) 
        closed_logits = self.closed_head(pooled_memory)

        answer_embeds = self.embedding(answer_ids)
        if self.training:
            answer_embeds = self.pre_head_dropout(answer_embeds)  
        # ─────────────────────────────────────────────────────────────────

        tgt_mask = nn.Transformer.generate_square_subsequent_mask(answer_ids.size(1)).to(pixel_values.device)
        output = self.decoder(tgt=answer_embeds, memory=memory, tgt_mask=tgt_mask)
        open_logits = self.fc_out(output)
        
        return closed_logits, open_logits

    @torch.no_grad()
    def generate(self, pixel_values, question_ids, question_types,
                 tokenizer, attention_mask=None, num_beams=3, max_len=32, do_sample=False, temperature=1.0, top_p=0.9):
        self.eval()
        batch_size = pixel_values.size(0)
        dev        = pixel_values.device
 
        vision_feats   = self.vision_encoder(pixel_values).last_hidden_state
        vision_feats   = self.vision_projection(vision_feats)
        bert_output    = self.bert_encoder(input_ids=question_ids, attention_mask=attention_mask)
        question_feats = bert_output.last_hidden_state
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            question_feats = question_feats * mask
        attn_output, _ = self.cross_attention(query=question_feats, key=vision_feats, value=vision_feats)
        memory = self.layer_norm(question_feats + attn_output)
 
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (memory * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            pooled = memory.mean(dim=1)
        pooled_memory = self.pre_head_dropout(pooled)
        closed_logits = self.closed_head(pooled_memory)
        closed_preds  = closed_logits.argmax(dim=-1)
 
        # ── NUOVO: confidence CLOSED = prob max dopo softmax ────────────────
        closed_probs       = torch.softmax(closed_logits, dim=-1)
        closed_confidence  = closed_probs.max(dim=-1).values  # (batch_size,)

        if do_sample:
            generated = torch.full((batch_size, 1), tokenizer.cls_token_id,
                                   dtype=torch.long, device=dev)
            seq_logp = torch.zeros(batch_size, device=dev)
            for _ in range(max_len):
                tgt_mask = nn.Transformer.generate_square_subsequent_mask(generated.size(1)).to(dev)
                out = self.decoder(tgt=self.embedding(generated), memory=memory, tgt_mask=tgt_mask)
                logits = self.fc_out(out[:, -1, :]) / max(temperature, 1e-6)
                probs = torch.softmax(logits, dim=-1)
                # top-p
                sp, si = torch.sort(probs, descending=True, dim=-1)
                cum = sp.cumsum(dim=-1)
                sp[cum - sp > top_p] = 0.0
                sp = sp / sp.sum(dim=-1, keepdim=True)
                nxt = si.gather(1, torch.multinomial(sp, 1))
                seq_logp += torch.log_softmax(logits, dim=-1).gather(1, nxt).squeeze(1)
                generated = torch.cat([generated, nxt], dim=1)
                if (nxt.squeeze(1) == tokenizer.sep_token_id).all():
                    break
            best_generated = generated
            open_confidence = torch.exp(seq_logp / max(1, generated.size(1)-1)).clamp(0,1)
            closed_confidence = torch.softmax(closed_logits, dim=-1).max(dim=-1).values
            closed_preds_idx = closed_logits.argmax(dim=-1)
            confidences = torch.zeros(batch_size, device=dev)
            for i in range(batch_size):
                if question_types[i] == 0:
                    pad = tokenizer.pad_token_id
                    row = torch.full((best_generated.size(1),), pad, device=dev, dtype=torch.long)
                    row[0] = tokenizer.cls_token_id
                    row[1] = IDX_TO_TOKEN.get(closed_preds_idx[i].item(), YES_TOKEN_ID)
                    row[2] = tokenizer.sep_token_id
                    best_generated[i] = row
                    confidences[i] = closed_confidence[i]
                else:
                    confidences[i] = open_confidence[i]
            return best_generated, confidences

        
 
        memory_expanded = memory.repeat_interleave(num_beams, dim=0)
        



        
        generated       = torch.full((batch_size * num_beams, 1),
                                     tokenizer.cls_token_id,
                                     dtype=torch.long).to(dev)
        beam_scores = torch.zeros((batch_size, num_beams)).to(dev)
        beam_scores[:, 1:] = -1e9
        beam_scores = beam_scores.view(-1)

        
 
        for _ in range(max_len):
            tgt_mask = nn.Transformer.generate_square_subsequent_mask(
                generated.size(1)
            ).to(dev)
            output            = self.decoder(tgt=self.embedding(generated),
                                             memory=memory_expanded,
                                             tgt_mask=tgt_mask)
            next_token_logits = self.fc_out(output[:, -1, :])
            next_token_probs  = torch.log_softmax(next_token_logits, dim=-1)
 
            next_scores = next_token_probs + beam_scores[:, None]
            next_scores = next_scores.view(batch_size,
                                           num_beams * next_token_probs.size(-1))
 
            topk_scores, topk_indices = torch.topk(next_scores, num_beams, dim=1)
            beam_ids  = topk_indices // next_token_probs.size(-1)
            token_ids = topk_indices %  next_token_probs.size(-1)
 
            new_generated = []
            for i in range(batch_size):
                for j in range(num_beams):
                    prev_idx = i * num_beams + beam_ids[i, j]
                    new_seq  = torch.cat([generated[prev_idx],
                                          token_ids[i, j].unsqueeze(0)])
                    new_generated.append(new_seq)
 
            generated   = torch.stack(new_generated)
            beam_scores = topk_scores.view(-1)
            if (token_ids == tokenizer.sep_token_id).all():
                break
 
        best_generated = generated.view(batch_size, num_beams, -1)[:, 0, :]
 
        # ── NUOVO: confidence OPEN = best beam score normalizzato ───────────
        # best_beam_score è una log-prob cumulativa → la normalizziamo
        # per lunghezza e la convertiamo in probabilità lineare
        best_beam_scores = beam_scores.view(batch_size, num_beams)[:, 0]
        real_len = torch.zeros(batch_size, device=dev)
        for i in range(batch_size):
            seq = best_generated[i]
            sep_pos = (seq == tokenizer.sep_token_id).nonzero()
            real_len[i] = (sep_pos[0].item() if len(sep_pos) > 0 else (seq != tokenizer.pad_token_id).sum().item())
        real_len = real_len.clamp(min=1)
        open_confidence = torch.exp(best_beam_scores / real_len)
        open_confidence = open_confidence.clamp(0.0, 1.0)
        # clamp in [0,1] per sicurezza numerica
 
        # ── Routing finale (identico all'originale) ──────────────────────────
        closed_preds_idx = closed_logits.argmax(dim=-1)  # 0 o 1

        confidences = torch.zeros(batch_size, device=dev)
        for i in range(batch_size):
            if question_types[i] == 0:   # CLOSED
                pred_idx   = closed_preds_idx[i].item()
                pred_token = IDX_TO_TOKEN.get(pred_idx, YES_TOKEN_ID)  # fallback a yes
                best_generated[i]    = tokenizer.pad_token_id
                best_generated[i, 0] = tokenizer.cls_token_id
                best_generated[i, 1] = pred_token
                best_generated[i, 2] = tokenizer.sep_token_id
                confidences[i]       = closed_confidence[i]
            else:                        # OPEN
                confidences[i]       = open_confidence[i]
 
        # ──  restituisce anche confidences ──────────────────────────
        return best_generated, confidences   



In [ ]:
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

vocab_size_value = len(tokenizer)
model = CustomMedVQAModel(vocab_size=vocab_size_value).to(device)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np

# -----------------------------------------------------------------------------
# STEP 1: Funzione di Costo (Loss Function)
# -----------------------------------------------------------------------------
class MedVQAMultiClassFocalLoss(nn.Module):
    def __init__(self, gamma=1.8803049874792026, ignore_index=-100, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction

    def forward(self, logits, targets, is_open=False):
        # Label smoothing differenziato: meno smoothing per chiuse (vocabolario ristretto yes/no)
        smoothing = 0.15 if is_open else 0.05

        ce_loss = F.cross_entropy(
            logits, targets,
            reduction='none',
            ignore_index=self.ignore_index,
            label_smoothing=smoothing
        )

        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

# -----------------------------------------------------------------------------
# STEP 2: Inizializzazione Ambiente, Modello e Ottimizzatori
# -----------------------------------------------------------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

learning_rate = 0.00019560708142748483
LOSS_ALPHA=0.35
weight_decay = 0.0001238513729886094
accumulation_steps = 4 
trainable_params = filter(lambda p: p.requires_grad, model.parameters())

# AdamW previene l'overfitting scollegando il decadimento dal momento
optimizer = torch.optim.AdamW(trainable_params, lr=learning_rate, weight_decay=weight_decay)
# Scheduler per abbattere il learning rate quando la loss si stabilizza
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.47369321060486275, patience=2)

# Istanziazione del criterio e dello Scaler per la Precisione Mista
criterion = MedVQAMultiClassFocalLoss(gamma=1.8803049874792026, ignore_index=tokenizer.pad_token_id).to(device)
scaler = torch.amp.GradScaler('cuda')


In [ ]:
def run_epoch(model, dataloader, optimizer, criterion, scaler, device,
              is_train=True, accumulation_steps=4):
    model.train() if is_train else model.eval()
    running_loss = 0.0
    context = torch.enable_grad() if is_train else torch.no_grad()

    if is_train:
        optimizer.zero_grad()

    with context:
        for step, batch in enumerate(dataloader):
            img     = batch['pixel_values'].to(device)
            q       = batch['input_ids'].to(device)
            a       = batch['labels'].to(device)
            q_types = batch['question_type'].to(device)

            with torch.amp.autocast('cuda'):
                attn_mask = batch['attention_mask'].to(device)
                closed_logits, open_logits = model(img, q, a[:, :-1], attention_mask=attn_mask)

                loss        = torch.tensor(0.0, device=device)
                closed_mask = (q_types == 0)
                open_mask   = (q_types == 1)

                n_closed = closed_mask.sum().float()
                n_open   = open_mask.sum().float()
                total    = n_closed + n_open + 1e-8

                if closed_mask.any():
                    target_closed = (a[closed_mask, 1] == NO_TOKEN_ID).long()
                    logits_closed = closed_logits[closed_mask]
                    loss_closed   = criterion(logits_closed, target_closed, is_open=False)
                    loss += (n_closed / total) * LOSS_ALPHA * loss_closed
                    

                if open_mask.any():
                    target_flat  = a[open_mask, 1:].contiguous().view(-1)
                    logits_flat  = open_logits[open_mask].contiguous().view(-1, open_logits.size(-1))
                    non_pad_mask = (target_flat != tokenizer.pad_token_id)
                    if non_pad_mask.any():
                        loss_open = criterion(
                            logits_flat[non_pad_mask],
                            target_flat[non_pad_mask],
                            is_open=True
                        )
                        loss += (n_open / total) * (1 - LOSS_ALPHA) * loss_open

                loss_scaled = loss / accumulation_steps

            if is_train:
                scaler.scale(loss_scaled).backward()

                if (step + 1) % accumulation_steps == 0 or (step + 1) == len(dataloader):
                    scaler.unscale_(optimizer)

                    # ── STAMPA GRADIENTI ──────────────────────────────────
                    #total_norm = 0.0
                    #for p in model.parameters():
                     #   if p.grad is not None:
                      #      total_norm += p.grad.detach().data.norm(2).item() ** 2
                    #total_norm = total_norm ** 0.5

                    #accum_step = (step + 1) // accumulation_steps

                    # ─────────────────────────────────────────────────────

                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.5)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            running_loss += loss.item()

    return running_loss / len(dataloader)

  

# -----------------------------------------------------------------------------
# STEP 4: Loop Principale con Early Stopping
# -----------------------------------------------------------------------------
epochs             = 40
best_val_loss      = float('inf')
patience_counter   = 0
early_stop_patience = 9

for epoch in range(epochs):
    train_loss = run_epoch(
        model, train_loader, optimizer, criterion, scaler, device,
        is_train=True, accumulation_steps=accumulation_steps
    )
    val_loss = run_epoch(
        model, val_loader, None, criterion, None, device,
        is_train=False, accumulation_steps=accumulation_steps
    )

    scheduler.step(val_loss)

    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_vqa_model.pth")
        print(">>> Miglior modello salvato.")
    else:
        patience_counter += 1
        if patience_counter >= early_stop_patience:
            print(f"Early stopping all'epoca {epoch+1}. Miglior val loss: {best_val_loss:.4f}")
            break

In [ ]:
import json, random, time
from tqdm import tqdm
 
def generate_two_responses(model, pixel_values, question_ids, question_types,
                            tokenizer, device, attention_mask=None):
    """
    Genera due risposte diverse per lo stesso input usando beam search
    con numero di beams diverso — serve a ottenere due candidati distinti
    da dare in pasto a Gemini per la preferenza.
    """
    model.eval()
    with torch.no_grad():
        # Risposta A: beam search standard (5 beams)
        ids_a, conf_a = model.generate(
            pixel_values, question_ids, question_types,
            tokenizer, attention_mask=attention_mask, num_beams=4, max_len=32
        )
        # Risposta B: beam search più stretta (2 beams) → diversa dall'A
        ids_b, conf_b = model.generate(
            pixel_values, question_ids, question_types,
            tokenizer, attention_mask=attention_mask, max_len=32,
            do_sample=True, temperature=0.9, top_p=0.9
        )
    text_a = tokenizer.batch_decode(ids_a, skip_special_tokens=True)
    text_b = tokenizer.batch_decode(ids_b, skip_special_tokens=True)
    return text_a, text_b, ids_a, ids_b
 
 
def gemini_preference(question, response_a, response_b, ans_type,
                       organ, q_type, ground_truth, gemini_model,
                       max_retries=3):
    """
    Chiede a Gemini quale tra le due risposte è clinicamente più accurata.
    Restituisce 'A' o 'B'. In caso di errore restituisce None (coppia scartata).
 
    Il ground truth è incluso nel prompt per ancorare il giudizio,
    ma Gemini deve comunque ragionare sulla qualità clinica — non solo
    sull'exact match con il ground truth.
    """
    prompt = f"""You are an expert radiologist evaluating two candidate answers
to a medical imaging question.
 
Question: {question}
Organ: {organ}
Question type: {q_type}
Answer type: {ans_type}
Ground truth (reference): {ground_truth}
 
Candidate A: {response_a}
Candidate B: {response_b}
 
Which candidate is clinically more accurate and complete?
- Consider medical correctness, terminology, and relevance to the question.
- If both are equally good or bad, prefer the one closer to the ground truth.
 
Reply ONLY with a single character: A or B."""
 
    for attempt in range(max_retries):
        try:
            _gemini_rate_limit()
            response = gemini_model.generate_content(prompt)
            choice = response.text.strip().upper()
            if choice.startswith("A"):
                return "A"
            if choice.startswith("B"):
                return "B"
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                wait = 60 / GEMINI_MAX_PER_MIN * (attempt + 1)
                print(f"  [Gemini] Rate limit, attendo {wait:.0f}s...")
                time.sleep(wait)
            else:
                print(f"  [Gemini] Errore: {e} → coppia scartata")
                return None
    return None
 
 
def collect_dpo_preferences(model, dataloader, tokenizer, device,
                             gemini_model, save_path="/kaggle/working/dpo_pairs.json",
                             max_samples=400):
    """
    Raccoglie coppie di preferenze (y_w, y_l) per il DPO.
 
    Per ogni batch:
      1. Genera due risposte A e B
      2. Chiede a Gemini quale preferisce
      3. Salva la coppia (y_w=preferita, y_l=non_preferita) con il contesto
 
    Parametri:
      max_samples : numero massimo di coppie da raccogliere.
                    Con 400 coppie e rate limit 15 req/min → ~27 minuti.
                   
 
    Il file JSON salvato ha questa struttura:
      [{"question": ..., "y_w_ids": [...], "y_l_ids": [...],
        "y_w_text": ..., "y_l_text": ..., "ans_type": ...}, ...]
    """
    pairs   = []
    n_total = 0
 
    print(f"Raccolta preferenze RLAIF (max {max_samples} coppie)...")
 
    model.eval()
    for batch in tqdm(dataloader):
        if n_total >= max_samples:
            break
 
        pixel_values     = batch['pixel_values'].to(device)
        question_ids     = batch['input_ids'].to(device)
        q_types_idx      = batch['question_type'].to(device)
        labels           = batch['labels']
        ans_types_str    = batch['ans_type_str']
        q_types_str      = batch['q_type_str']
        organs_str       = batch['organ_str']
        original_qs      = batch['original_q']
        dataset_idxs = batch['dataset_idx']
        attn_mask = batch['attention_mask'].to(device)

        text_a, text_b, ids_a, ids_b = generate_two_responses(
            model, pixel_values, question_ids, q_types_idx, tokenizer, device, attention_mask=attn_mask
        )
        labels_text = tokenizer.batch_decode(labels, skip_special_tokens=True)
 
        for i in range(len(text_a)):
            if n_total >= max_samples:
                break
 
            ra = text_a[i].strip().lower()
            rb = text_b[i].strip().lower()
 
            # Salta coppie identiche — non portano info per il DPO
            if ra == rb:
                continue
 
            ans_type = ans_types_str[i]
            choice   = gemini_preference(
                question    = original_qs[i],
                response_a  = ra,
                response_b  = rb,
                ans_type    = ans_type,
                organ       = organs_str[i],
                q_type      = q_types_str[i],
                ground_truth= labels_text[i].strip().lower(),
                gemini_model= gemini_model,
            )
            if choice is None:
                continue  # Gemini ha fallito → scarta
 
            # y_w = preferita, y_l = non preferita
            if choice == "A":
                y_w_ids, y_l_ids = ids_a[i].tolist(), ids_b[i].tolist()
                y_w_text, y_l_text = ra, rb
            else:
                y_w_ids, y_l_ids = ids_b[i].tolist(), ids_a[i].tolist()
                y_w_text, y_l_text = rb, ra
 
            pairs.append({
                "question"   : original_qs[i],
                "ans_type"   : ans_type,
                "organ"      : organs_str[i],
                "q_type"     : q_types_str[i],
                "input_ids"  : question_ids[i].tolist(),
                "dataset_idx": int(dataset_idxs[i]),
                "pixel_values_shape": list(pixel_values[i].shape),
                "y_w_ids"    : y_w_ids,
                "y_l_ids"    : y_l_ids,
                "y_w_text"   : y_w_text,
                "y_l_text"   : y_l_text,
                "gt_text"    : labels_text[i].strip().lower(),
            })
            n_total += 1
            if n_total % 50 == 0:
                with open(save_path, "w") as f:
                    json.dump(pairs, f, indent=2)
                print(f"  Checkpoint: {n_total} coppie salvate.")
 
    with open(save_path, "w") as f:
        json.dump(pairs, f, indent=2)
 
    print(f"\nRaccolta completata: {len(pairs)} coppie salvate in {save_path}")
    return pairs
 
 
# --- ESECUZIONE: raccolta preferenze sul training set ---
# Con max_samples=400 e rate limit 15 req/min → ~27 minuti
dpo_pairs = collect_dpo_preferences(
    model        = model,
    dataloader   = train_loader,
    tokenizer    = tokenizer,
    device       = device,
    gemini_model = gemini_model,
    save_path    = "/kaggle/working/dpo_pairs.json",
    max_samples  = 400,
)

In [ ]:
from torch.utils.data import Dataset as TorchDataset, DataLoader as TorchDataLoader
import copy
 
class DPOPairsDataset(TorchDataset):
    """
    Dataset PyTorch per le coppie di preferenze DPO.
    Carica le coppie salvate da collect_dpo_preferences e le restituisce
    come tensori pronti per il training.
    """
    def __init__(self, pairs_path, tokenizer, max_len=32):
        import os
        if not os.path.exists(pairs_path):
            raise FileNotFoundError(f"File DPO non trovato: {pairs_path}. Esegui prima collect_dpo_preferences.")
        with open(pairs_path) as f:
            self.pairs = json.load(f)
        if len(self.pairs) == 0:
            raise ValueError(f"dpo_pairs.json è vuoto. collect_dpo_preferences ha prodotto 0 coppie.")
        self.tokenizer = tokenizer
        self.max_len   = max_len
 
    def __len__(self):
        return len(self.pairs)
 
    def __getitem__(self, idx):
        pair = self.pairs[idx]
 
        def pad_or_trim(ids):
            ids = ids[:self.max_len]
            pad = self.max_len - len(ids)
            return ids + [self.tokenizer.pad_token_id] * pad

        def pad_or_trim_q(ids, max_len=512):
            ids = ids[:max_len]
            return ids + [self.tokenizer.pad_token_id] * (max_len - len(ids))
 
        return {
            "input_ids": torch.tensor(pad_or_trim_q(pair["input_ids"]), dtype=torch.long),
            "y_w_ids"   : torch.tensor(pad_or_trim(pair["y_w_ids"]), dtype=torch.long),
            "y_l_ids"   : torch.tensor(pad_or_trim(pair["y_l_ids"]), dtype=torch.long),
            "ans_type"  : pair["ans_type"],
            "dataset_idx": pair["dataset_idx"],
        }
 
 
def compute_log_probs(model, pixel_values_dummy, question_ids,
                      answer_ids, device, attention_mask=None):
    """
    Calcola la log-probabilità media della sequenza answer_ids
    dato il contesto (question_ids).
 
    Usa il forward del modello esistente con teacher forcing:
      - input decoder: answer_ids[:, :-1]
      - target:        answer_ids[:, 1:]
 
    Restituisce un tensore (batch_size,) con la log-prob media per sequenza.
 
    NOTA: pixel_values_dummy è un tensore zero — nel DPO confrontiamo
    le probabilità relative tra π e π_ref, quindi la componente visiva
    si cancella nel rapporto. Questo semplifica molto il calcolo e
    riduce la memoria GPU necessaria.
    """
    # Forward con teacher forcing
    _, open_logits = model(pixel_values_dummy, question_ids, answer_ids[:, :-1], attention_mask=attention_mask)
    # open_logits: (batch, seq_len-1, vocab_size)
 
    targets = answer_ids[:, 1:].contiguous()  # (batch, seq_len-1)
 
    # Log-prob per ogni token
    log_probs = F.log_softmax(open_logits, dim=-1)  # (batch, seq_len-1, vocab)
 
    # Raccogli la log-prob del token corretto per ogni posizione
    # gather: prende dalla dim vocabolario l'indice corretto
    token_log_probs = log_probs.gather(
        dim=2,
        index=targets.unsqueeze(2).clamp(min=0)
    ).squeeze(2)  # (batch, seq_len-1)
 
    # Maschera i token di padding
    #pad_mask = (targets != model.embedding.weight.shape[0] - 1).float()
    # Usiamo pad_token_id=0 come fallback — ignora posizioni di padding
    pad_mask = (targets != tokenizer.pad_token_id).float()
 
    # Media per sequenza (normalizzata per lunghezza)
    seq_log_probs = (token_log_probs * pad_mask).sum(dim=1) 
    return seq_log_probs  # (batch_size,)
 
 
class DPOLoss(nn.Module):
    """
    Loss DPO (Direct Preference Optimization).
 
    L = -log σ( β * (log π(y_w)/π_ref(y_w) - log π(y_l)/π_ref(y_l)) )
 
    Parametri:
      beta : controlla quanto il modello può discostarsi dal riferimento.
             Valori tipici: 0.05-0.5. Più alto → meno regolarizzazione.
    """
    def __init__(self, beta=0.1):
        super().__init__()
        self.beta = beta
 
    def forward(self, log_pi_yw, log_pi_yl, log_ref_yw, log_ref_yl):
        # Rapporti log π/π_ref
        ratio_w = log_pi_yw - log_ref_yw
        ratio_l = log_pi_yl - log_ref_yl
 
        # Reward implicito
        reward  = self.beta * (ratio_w - ratio_l)
 
        # Loss = -log σ(reward)
        loss    = -F.logsigmoid(reward).mean()
        return loss
 
 

In [ ]:
DPO_BETA       = 0.1    # temperatura DPO
DPO_LR         = 1e-5   # learning rate molto basso per fine-tuning stabile
DPO_EPOCHS     = 3      # 3 epoche sono sufficienti con 400 coppie
DPO_BATCH_SIZE = 4      # batch piccolo per stabilità
 
# --- 1. Carica il modello migliore dal checkpoint SFT ---
model.load_state_dict(torch.load("/kaggle/working/best_vqa_model.pth",
                                  map_location=device))
model.to(device)
 
# --- 2. Crea il modello di riferimento (π_ref) — completamente frozen ---
ref_model = copy.deepcopy(model)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False
print("Modello di riferimento (π_ref) creato e congelato.")
 
# --- 3. Congela tutto tranne il decoder nel modello attivo (π) ---
for param in model.parameters():
    param.requires_grad = False
 
# Scongela SOLO i layer del decoder Transformer
for param in model.decoder.parameters():
    param.requires_grad = True
 
# Scongela anche fc_out (condivide pesi con embedding via weight tying,
# ma i gradienti scorrono solo attraverso fc_out durante il DPO)
#model.fc_out.weight = nn.Parameter(model.embedding.weight.data.clone())
#for param in model.fc_out.parameters():
    #param.requires_grad = True
 
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parametri aggiornabili nel DPO: {n_trainable:,} "
      f"(solo decoder + fc_out)")
 
# --- 4. Dataset e dataloader DPO ---
dpo_dataset = DPOPairsDataset(
    pairs_path = "/kaggle/working/dpo_pairs.json",
    tokenizer  = tokenizer,
    max_len    = 32,
)
dpo_loader = TorchDataLoader(
    dpo_dataset,
    batch_size = DPO_BATCH_SIZE,
    shuffle    = True,
)
print(f"Dataset DPO: {len(dpo_dataset)} coppie | "
      f"{len(dpo_loader)} batch per epoca")
 
# --- 5. Ottimizzatore e loss DPO ---
dpo_criterion = DPOLoss(beta=DPO_BETA)
dpo_optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = DPO_LR,
    weight_decay = 1e-4,
)
dpo_scaler = torch.amp.GradScaler('cuda')
 
# --- 6. Loop DPO ---
print("\nInizio fine-tuning DPO...")
best_dpo_loss = float('inf')
 
for epoch in range(DPO_EPOCHS):
    model.train()
    running_loss = 0.0
 
    for batch in tqdm(dpo_loader, desc=f"DPO Epoch {epoch+1}/{DPO_EPOCHS}"):
        q_ids  = batch["input_ids"].to(device)   # (B, 512)
        q_attn_mask = (q_ids != tokenizer.pad_token_id).long().to(device)
        y_w    = batch["y_w_ids"].to(device)      # (B, 32)
        y_l    = batch["y_l_ids"].to(device)      # (B, 32)
        B      = q_ids.size(0)
 
        q_ids  = batch["input_ids"].to(device)
        q_attn_mask = (q_ids != tokenizer.pad_token_id).long().to(device)
        y_w    = batch["y_w_ids"].to(device)
        y_l    = batch["y_l_ids"].to(device)
        B      = q_ids.size(0)

        idxs = batch["dataset_idx"].tolist() if torch.is_tensor(batch["dataset_idx"]) else list(batch["dataset_idx"])
        real_pixels = torch.stack([train_set[j]["pixel_values"] for j in idxs]).to(device)
 
        with torch.amp.autocast('cuda'):
            # Log-prob dal modello attivo π
            log_pi_yw  = compute_log_probs(model,     real_pixels, q_ids, y_w, device, attention_mask=q_attn_mask)
            log_pi_yl  = compute_log_probs(model,     real_pixels, q_ids, y_l, device, attention_mask=q_attn_mask)
 
            # Log-prob dal modello di riferimento π_ref (no grad)
            with torch.no_grad():
                log_ref_yw = compute_log_probs(ref_model, real_pixels, q_ids, y_w, device, attention_mask=q_attn_mask)
                log_ref_yl = compute_log_probs(ref_model, real_pixels, q_ids, y_l, device, attention_mask=q_attn_mask)
 
            loss = dpo_criterion(log_pi_yw, log_pi_yl, log_ref_yw, log_ref_yl)
 
        dpo_scaler.scale(loss).backward()
        # Gradient clipping per stabilità
        dpo_scaler.unscale_(dpo_optimizer)
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()),
            max_norm=1.0
        )
        dpo_scaler.step(dpo_optimizer)
        dpo_scaler.update()
        dpo_optimizer.zero_grad()
 
        running_loss += loss.item()
 
    avg_loss = running_loss / len(dpo_loader)
    print(f"Epoch {epoch+1:02d} | DPO Loss: {avg_loss:.4f}")
 
    if avg_loss < best_dpo_loss:
        best_dpo_loss = avg_loss
        torch.save(model.state_dict(), "/kaggle/working/best_vqa_model_dpo.pth")
        print("  >>> Miglior checkpoint DPO salvato.")
 
print("\nFine-tuning DPO completato.")
print("Per valutare il modello DPO, esegui evaluate_model_on_test_set()")
print("usando il checkpoint: /kaggle/working/best_vqa_model_dpo.pth")

In [ ]:
import pandas as pd
import torch
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from tqdm import tqdm
from bert_score import BERTScorer 

def gemini_refine(question, raw_answer, ans_type, rag_context, kb_context,
                  organ, q_type, gemini_model, max_retries=3):
    """
    Chiama Gemini per raffinare la risposta grezza del modello VQA.
 
    Il prompt è strutturato in modo diverso per CLOSED e OPEN:
      - CLOSED: chiede di scegliere tra yes/no basandosi sul contesto
      - OPEN  : chiede di riscrivere la risposta in modo più preciso
 
    Gestisce il rate limit (15 req/min) con retry automatico.
    """
 
    # Prompt differenziato per tipo di domanda
    if ans_type == "CLOSED":
        prompt = f"""You are a radiology expert. Answer the following yes/no question based on the provided context.
 
Medical Context:
{rag_context}
{kb_context}
 
Organ: {organ}
Question type: {q_type}
Question: {question}
Initial model answer: {raw_answer}
 
Instructions:
- Answer ONLY with 'yes' or 'no'
- Use the context and your medical knowledge to correct the initial answer if needed
- Do not add any explanation
- think carefully at each step

Answer:"""
 
    else:  # OPEN
        prompt = f"""You are a radiology expert. Refine the following answer to a radiology question.
 
Medical Context:
{rag_context}
{kb_context}
 
Organ: {organ}
Question type: {q_type}
Question: {question}
Initial model answer: {raw_answer}
 
Instructions:
- Provide a concise, clinically accurate answer (max 30 words)
- Use proper medical terminology
- If the initial answer is already correct, return it as-is
- Do not add explanations or preambles
- Think carefully at each step
Refined answer:"""
 
    # Retry con backoff per gestire rate limit
    for attempt in range(max_retries):
        try:
            _gemini_rate_limit()
            response = gemini_model.generate_content(prompt)
            refined  = response.text.strip().lower()
            # Pulizia risposta: rimuovi eventuali prefissi indesiderati
            for prefix in ["refined answer:", "answer:", "response:"]:
                if refined.startswith(prefix):
                    refined = refined[len(prefix):].strip()
            return refined
        except Exception as e:
            if "429" in str(e) or "quota" in str(e).lower():
                # Rate limit → aspetta e riprova
                wait = 60 / GEMINI_MAX_PER_MIN * (attempt + 1)  # backoff progressivo
                print(f"  [Gemini] Rate limit, attendo {wait:.0f}s...")
                time.sleep(wait)
            else:
                print(f"  [Gemini] Errore: {e} → uso risposta grezza")
                return raw_answer  # fallback sicuro
 
    return raw_answer  # fallback dopo tutti i retry


scorer = BERTScorer(
    model_type='emilyalsentzer/Bio_ClinicalBERT',
    num_layers=9, device=device, lang="en"
)
scorer._tokenizer.model_max_length = 512
 
def evaluate_model_on_test_set(model, test_loader, tokenizer, device,
                                rag_collection=None, rag_encoder=None,
                                kb_collection=None, kb_encoder=None, scorer=None):
    model.eval()
    records = []
 
    print(f"Avvio inferenza su {len(test_loader.dataset)} campioni del Test Set...")
 
    with torch.no_grad():
        for batch in tqdm(test_loader):
            pixel_values     = batch['pixel_values'].to(device)
            question_ids     = batch['input_ids'].to(device)
            labels           = batch['labels']
            question_types_idx = batch['question_type'].to(device)
            ans_types_str    = batch['ans_type_str']
            q_types_str      = batch['q_type_str']
            organs_str       = batch['organ_str']
            original_qs      = batch['original_q']
 
            # ── MODIFICA: unpack di (ids, confidences) ──────────────────────
            attn_mask = batch['attention_mask'].to(device)
            generated_ids, confidences = model.generate(
                pixel_values=pixel_values,
                question_ids=question_ids,
                question_types=question_types_idx,
                tokenizer=tokenizer,
                attention_mask=attn_mask,
                num_beams=5,
                max_len=32
            )
 
            preds_text  = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
            labels_text = tokenizer.batch_decode(labels,        skip_special_tokens=True)
 
            for i in range(len(preds_text)):
                raw_pred   = preds_text[i].strip().lower()
                ans_type   = ans_types_str[i]
                confidence = confidences[i].item()
                refined_by_gemini = False
 
                # ── NUOVO: routing Gemini POST ───────────────────────────────
                threshold = (CONFIDENCE_THRESHOLD_CLOSED
                             if ans_type == "CLOSED"
                             else CONFIDENCE_THRESHOLD_OPEN)
 
                if confidence < threshold:
                    # Recupera il contesto RAG per questo campione
                    # (lo usiamo nel prompt Gemini per dare più contesto)
                    organ   = organs_str[i]
                    q_type  = q_types_str[i]
                    orig_q  = original_qs[i]
                    ctx_q   = f"[{organ} | {q_type}] {orig_q}"
 
                    rag_ctx = ""
                    kb_ctx  = ""
                    if rag_collection is not None and rag_encoder is not None:
                        rag_ctx = retrieve_similar_pairs(
                            ctx_q, organ, ans_type,
                            rag_collection, rag_encoder, k=3
                        )
                    if kb_collection is not None and kb_encoder is not None:
    
                        kb_ctx = retrieve_kb_context(ctx_q, kb_collection, kb_encoder, k=2)

 
                    final_pred = gemini_refine(
                        question    = orig_q,
                        raw_answer  = raw_pred,
                        ans_type    = ans_type,
                        rag_context = rag_ctx,
                        kb_context  = kb_ctx,
                        organ       = organ,
                        q_type      = q_type,
                        gemini_model= gemini_model,
                    )
                    refined_by_gemini = (final_pred.strip().lower() != raw_pred)
                else:
                    final_pred = raw_pred
 
                records.append({
                    "Organo":             organs_str[i],
                    "Categoria_Domanda":  q_types_str[i],
                    "Tipo_Risposta":      ans_type,
                    "Domanda":            original_qs[i],
                    "Predizione_Grezza":  raw_pred,           # ← NUOVO
                    "Predizione":         final_pred,         # ← ora può essere raffinata
                    "Ground_Truth":       labels_text[i].strip().lower(),
                    "Confidence":         round(confidence, 4), # ← NUOVO
                    "Refined_By_Gemini":  refined_by_gemini,  # ← NUOVO
                })
 
    df = pd.DataFrame(records)
 
    # ── Metriche identiche all'originale ────────────────────────────────────
    df['Exact_Match'] = (df['Predizione'] == df['Ground_Truth']).astype(int)
 
    chencherry = SmoothingFunction()
    def calc_bleu(row):
        ref = row['Ground_Truth'].split()
        hyp = row['Predizione'].split()
        return sentence_bleu([ref], hyp, weights=(1,0,0,0),
                             smoothing_function=chencherry.method1)
    df['BLEU1'] = df.apply(calc_bleu, axis=1)
 
    print("\nCalcolo Metriche Semantiche (BERTScore) in corso...")
    if scorer is None:
        scorer = BERTScorer(
            model_type='emilyalsentzer/Bio_ClinicalBERT',
            num_layers=9, device=device, lang="en"
        )
        scorer._tokenizer.model_max_length = 512
    P, R, F1 = scorer.score(df['Predizione'].tolist(), df['Ground_Truth'].tolist())
    df['BERTScore_F1'] = F1.numpy()
 
    # ── Report identico all'originale + stats Gemini ────────────────────────
    print("\n" + "="*75)
    print(" REPORT GLOBALE DEL MODELLO VQA")
    print("="*75)
    print(f"Totale Campioni Analizzati:   {len(df)}")
    print(f"Exact Match Accuracy Globale: {df['Exact_Match'].mean():.4f}")
    print(f"BLEU-1 Globale:               {df['BLEU1'].mean():.4f}")
    print(f"BERTScore F1 Globale:         {df['BERTScore_F1'].mean():.4f}")
 
    # ── NUOVO: stats Gemini ──────────────────────────────────────────────────
    n_refined = df['Refined_By_Gemini'].sum()
    print(f"\nCampioni raffinati da Gemini: {n_refined}/{len(df)} "
          f"({100*n_refined/len(df):.1f}%)")
    if n_refined > 0:
        df_refined     = df[df['Refined_By_Gemini']]
        df_not_refined = df[~df['Refined_By_Gemini']]
        print(f"  BERTScore F1 campioni raffinati    : "
              f"{df_refined['BERTScore_F1'].mean():.4f}")
        print(f"  BERTScore F1 campioni non raffinati: "
              f"{df_not_refined['BERTScore_F1'].mean():.4f}")
 
    print("\n" + "="*75)
    print(" PRESTAZIONI PER TIPOLOGIA DI DOMANDA (Aperta vs Chiusa)")
    print("="*75)
    risultati_tipo = df.groupby('Tipo_Risposta')[
        ['Exact_Match', 'BLEU1', 'BERTScore_F1']
    ].mean().round(4)
    print(risultati_tipo.to_markdown())
 
    print("\n" + "="*75)
    print(" PRESTAZIONI PER ORGANO ANATOMICO")
    print("="*75)
    organ_counts  = df['Organo'].value_counts()
    valid_organs  = organ_counts[organ_counts >= 5].index
    df_organs     = df[df['Organo'].isin(valid_organs)]
    risultati_organo = df_organs.groupby('Organo')[
        ['Exact_Match', 'BLEU1', 'BERTScore_F1']
    ].mean().round(4)
    print(risultati_organo.to_markdown())
 
    print("\n" + "="*75)
    print(" ERROR ANALYSIS: I 5 peggiori errori sulle domande APERTE")
    print("="*75)
    errors = df[
        (df['Tipo_Risposta'] == 'OPEN') & (df['Exact_Match'] == 0)
    ].sort_values(by='BERTScore_F1').head(5)
    for idx, row in errors.iterrows():
        print(f"[{row['Organo']} | {row['Categoria_Domanda']}] "
              f"Q: {row['Domanda']}")
        print(f"   => Grezza: {row['Predizione_Grezza']} "
              f"| Finale: {row['Predizione']} "
              f"| Corretta: {row['Ground_Truth']} "
              f"| F1: {row['BERTScore_F1']:.4f} "
              f"| Conf: {row['Confidence']:.4f}\n")
 
    print("\n" + "="*75)
    print(" CASI DI SUCCESSO: 5 Risposte Mediche Articolate Perfette")
    print("="*75)
    successes = df[
        (df['Tipo_Risposta'] == 'OPEN') &
        (df['Exact_Match'] == 1) &
        (df['Ground_Truth'].str.split().str.len() >= 2)
    ].head(5)
    if successes.empty:
        print("Nessun caso di successo articolato trovato.")
    else:
        for idx, row in successes.iterrows():
            print(f"[{row['Organo']} | {row['Categoria_Domanda']}] "
                  f"Q: {row['Domanda']}")
            print(f"   => Modello: {row['Predizione']}\n")
 
    return df
 
 
# =============================================================================
 
print("\n" + "="*60)
print("VALUTAZIONE MODELLO SFT (pre-DPO)")
print("="*60)
model.load_state_dict(torch.load("/kaggle/working/best_vqa_model.pth", map_location=device))
model.to(device)
model.eval()
df_sft = evaluate_model_on_test_set(model, test_loader, tokenizer, device,
                                     rag_collection=rag_collection,
                                     rag_encoder=rag_encoder,
                                     kb_collection=kb_collection,
                                     kb_encoder=rag_encoder,scorer=scorer)

print("\n" + "="*60)
print("VALUTAZIONE MODELLO DPO (post fine-tuning)")
print("="*60)
model.load_state_dict(torch.load("/kaggle/working/best_vqa_model_dpo.pth", map_location=device))
model.to(device)
model.eval()
df_dpo = evaluate_model_on_test_set(model, test_loader, tokenizer, device,
                                     rag_collection=rag_collection,
                                     rag_encoder=rag_encoder,
                                     kb_collection=kb_collection,
                                     kb_encoder=rag_encoder,scorer=scorer)

print("\n" + "="*60)
print("DELTA DPO vs SFT")
print("="*60)
print(f"BERTScore F1: {df_dpo['BERTScore_F1'].mean():.4f} vs {df_sft['BERTScore_F1'].mean():.4f} "
      f"(Δ {df_dpo['BERTScore_F1'].mean() - df_sft['BERTScore_F1'].mean():+.4f})")
print(f"Exact Match:  {df_dpo['Exact_Match'].mean():.4f} vs {df_sft['Exact_Match'].mean():.4f} "
      f"(Δ {df_dpo['Exact_Match'].mean() - df_sft['Exact_Match'].mean():+.4f})")